In [2]:
import os
import math
import unicodedata
import torch
import torch.nn as nn
import numpy as np # Ensure numpy is installed: pip install numpy

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

MODEL_PATH = 'bert_diac_pytorch_with_der.pt' 

MAXLEN = 500
EMBED_DIM = 64
NHEAD = 4
NUM_LAYERS = 2
FF_DIM = 256
DROPOUT = 0.1

# Special Tokens
PLACEHOLDER = '<NONAR>'
PAD_TOKEN = '<PAD>'
SOS_TOKEN = '<SOS>'
EOS_TOKEN = '<EOS>'
UNK_TOKEN = '<UNK>'
SPACE_TOKEN = '<SPACE>'

Using device: cpu


In [5]:
# Cell 11: Model Definition (Matches Training Code)

try:
    from transformers import BertConfig, BertModel
except ImportError:
    raise ImportError("The 'transformers' library is not installed. Please run '!pip install transformers'.")

import torch.nn as nn

class Bert_Diac(nn.Module):
    def __init__(self, vocab_size, emb_dim, nhead, num_layers, ff_dim, num_labels, pad_idx, dropout=0.1):
        super().__init__()
        self.pad_idx = pad_idx

        # Build BERT config
        config = BertConfig(
            vocab_size=vocab_size,
            hidden_size=emb_dim,       # Must be divisible by nhead
            num_attention_heads=nhead,
            num_hidden_layers=num_layers,
            intermediate_size=ff_dim,
            hidden_dropout_prob=dropout,
            attention_probs_dropout_prob=dropout,
            type_vocab_size=1,
            max_position_embeddings=MAXLEN + 50, # Ensures coverage for SOS/EOS
            pad_token_id=pad_idx
        )

        # Initialize BERT from scratch (Random Weights)
        self.bert = BertModel(config)

        # Projection to diacritic label space
        self.out = nn.Linear(config.hidden_size, num_labels)

    def forward(self, x, src_key_padding_mask=None):
        # x: [batch, seq_len] token ids
        
        # Create attention mask for BERT (1 for valid token, 0 for pad)
        # We ignore src_key_padding_mask and calculate our own for BERT format
        attention_mask = (x != self.pad_idx).long()
        
        # Pass to BERT
        outputs = self.bert(input_ids=x, attention_mask=attention_mask)
        
        # Get last hidden state
        last_hidden = outputs.last_hidden_state
        
        # Project to labels
        logits = self.out(last_hidden)
        return logits

In [6]:
# Cell 12: Load Model

# 1. Load the checkpoint dictionary
print(f"Loading checkpoint from {MODEL_PATH}...")
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model file not found at {MODEL_PATH}. Check the path.")

ckpt = torch.load(MODEL_PATH, map_location=device)

# 2. Retrieve vocabularies
char2idx = ckpt['char2idx']
diac2idx = ckpt['diac2idx']
idx2diac = {v: k for k, v in diac2idx.items()}

vocab_size = len(char2idx)
num_labels = len(diac2idx)
pad_idx = char2idx[PAD_TOKEN]

# 3. Initialize Model (Using Bert_Diac now)
model = Bert_Diac(
    vocab_size=vocab_size, 
    emb_dim=EMBED_DIM,      # 64
    nhead=NHEAD,            # 4
    num_layers=NUM_LAYERS,  # 2
    ff_dim=FF_DIM,          # 256
    num_labels=num_labels, 
    pad_idx=pad_idx, 
    dropout=DROPOUT
)

# 4. Load Weights
try:
    model.load_state_dict(ckpt['model_state_dict'])
    print("Weights loaded successfully.")
except RuntimeError as e:
    print("Error loading weights:", e)

model.to(device)
model.eval()

Loading checkpoint from bert_diac_pytorch_with_der.pt...


C:\Users\maria\AppData\Local\Temp\ipykernel_9580\1807257465.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(MODEL_PATH, map_location=device)


Weights loaded successfully.


Bert_Diac(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(45, 64, padding_idx=0)
      (position_embeddings): Embedding(550, 64)
      (token_type_embeddings): Embedding(1, 64)
      (LayerNorm): LayerNorm((64,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-1): 2 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=64, out_features=64, bias=True)
              (key): Linear(in_features=64, out_features=64, bias=True)
              (value): Linear(in_features=64, out_features=64, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=64, out_features=64, bias=True)
              (LayerNorm): LayerNorm((64,), eps=1e-12, elementwise_affine=True)
        

In [7]:
def is_arabic_letter(ch):
    if len(ch) != 1: return False
    o = ord(ch)
    return (0x0600 <= o <= 0x06FF) or (ch == "ـ")

def strip_diacritics(s):
    # Remove existing diacritics to simulate raw input
    return "".join(c for c in s if unicodedata.category(c) != "Mn")

def label_to_mark(lab):
    if not lab or lab in ["<NONE>", "<PAD_LABEL>", "<OTHER>"]:
        return ""
    return lab

def prepare_inference_raw_line(raw_line):
    # Separate Arabic chars from Spaces/Non-Arabic for reconstruction later
    tokens = []
    original_nonar = []
    for ch in raw_line:
        if is_arabic_letter(ch):
            tokens.append(ch)
        elif ch.isspace():
            tokens.append(SPACE_TOKEN)
            original_nonar.append((len(tokens) - 1, ch))
        else:
            tokens.append(PLACEHOLDER)
            original_nonar.append((len(tokens) - 1, ch))
    return tokens, original_nonar

def infer_and_reconstruct(raw_line, model, device):
    # 1. Preprocess
    raw_clean = strip_diacritics(raw_line)
    tokens, restore_map = prepare_inference_raw_line(raw_clean)
    
    if not tokens: return ""

    # 2. Tokenize
    seq = [SOS_TOKEN] + tokens + [EOS_TOKEN]
    x_ids = [char2idx.get(t, char2idx[UNK_TOKEN]) for t in seq]
    
    # 3. Pad to MAXLEN
    original_len = len(x_ids)
    if len(x_ids) < MAXLEN:
        x_ids += [char2idx[PAD_TOKEN]] * (MAXLEN - len(x_ids))
    else:
        x_ids = x_ids[:MAXLEN]
        original_len = MAXLEN

    # 4. Prepare Tensor & Mask
    x_tensor = torch.tensor([x_ids], dtype=torch.long).to(device)
    
    # Create mask: True where value is pad_idx
    src_key_padding_mask = (x_tensor == char2idx[PAD_TOKEN])

    # 5. Model Prediction
    with torch.no_grad():
        logits = model(x_tensor, src_key_padding_mask=src_key_padding_mask)
        pred_ids = torch.argmax(logits, dim=-1).cpu().numpy()[0]

    # 6. Map IDs to Diacritics
    pred_labels = [idx2diac.get(i, "<NONE>") for i in pred_ids]
    
    # Trim SOS/EOS/PAD
    # We started at index 1 (skipping SOS), and take length matching the tokens
    pred_trim = pred_labels[1 : 1 + len(tokens)]

    # 7. Reconstruction
    out_chars = []
    for i, tok in enumerate(tokens):
        if tok == SPACE_TOKEN:
            out_chars.append(" ")
        elif tok == PLACEHOLDER:
            # Find original non-arabic char
            orig = next((ch for pos, ch in restore_map if pos == i), "?")
            out_chars.append(orig)
        else:
            # Arabic char + predicted diacritic
            lab = pred_trim[i] if i < len(pred_trim) else ""
            mark = label_to_mark(lab)
            out_chars.append(tok + mark)
            
    return "".join(out_chars)

In [8]:
test_file_path = 'test_no_diacritics.txt' 

if os.path.exists(test_file_path):
    print("-" * 30)
    print(f"Reading first 5 lines from {test_file_path}:")
    print("-" * 30)
    with open(test_file_path, 'r', encoding='utf8') as f:
        lines = [next(f) for _ in range(5)]
        
    for line in lines:
        line = line.strip()
        if not line: continue
        result = infer_and_reconstruct(line, model, device)
        print(f"RAW:  {line}")
        print(f"PRED: {result}")
        print()

------------------------------
Reading first 5 lines from test_no_diacritics.txt:
------------------------------
RAW:  ليس للوكيل بالقبض أن يبرأ المدين أو يهب الدين له أو يأخذ رهنا من المدين في مقابل الدين أو يقبل إحالته على شخص آخر لكن له أن يأخذ كفيلا لكن ليس له أن يأخذ كفيلا بشرط براءة الأصيل انظر المادة ( 648 ) ( الأنقروي ، الطحطاوي وصرة الفتاوى ، البحر ) .
PRED: لَيْسَ لِلْوَكِيلِ بِالْقَبْضِ أَنْ يَبْرَأَ الْمَدِينِ أَوْ يُهَبُ الدَّيْنِ لَهُ أَوْ يَأْخُذُ رَهْنًا مِنْ الْمَدِينِ فِي مُقَابِلُ الدَّيْنِ أَوْ يَقْبَلُ إحَالَتِهِ عَلَى شَخْصٍ آخَرَ لَكِنْ لَهُ أَنْ يَأْخُذَ كَفِيلًا لَكِنْ لَيْسَ لَهُ أَنْ يَأْخُذَ كَفِيلًا بِشَرْطِ بِرَاءَةِ الْأَصِيلِ انْظُرْ الْمَادَةِ ( 648 ) ( الْأَنْقَرِوِيَ ، الطَّحْطَاوِيُّ وَصِرَةُ الْفِتَاوَى ، الْبَحْرِ ) .

RAW:  ( قوله ويقع في بعض النسخ بمنفعة ومعين ) أي : أوصى بمجموع شيئين بمنفعة شيء وبمعين وقوله وليس ذلك بصحيح كأن عدم الصحة من جهة أن هذه المسألة فيها نص بهذا الحكم الذي أشار إليه المصنف بقوله وإن أوصى بمنفعة معين وبعض شيوخنا علل عدم ا